In [2]:
# 모델 매개변수 최적화
# 모델 학습 반복 과정 : 추측 -> 추측과 정답 사이의 손실 계산 -> 손실에 대한 변화율 계산 -> 변화율 기반 최적화

In [ ]:
import torch
from torch import nn 
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
  root="data",
  train=True,
  download=True,
  transform=ToTensor()
)

test_data = datasets.FashionMNIST(
  root="data",
  train=False,
  download=True,
  transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
      nn.Linear(28 * 28, 512),
      nn.ReLU(),
      nn.Linear(512, 512),
      nn.ReLU(),
      nn.Linear(512, 10),
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits
  
model = NeuralNetwork()

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [4]:
# 하이퍼파라미터: 모델 최적화 과정을 제어할 수 있는 매개변수
# 에폭(epoch) 수 : 데이터셋을 반복하는 횟수
# 배치 크기(batch_size) : 매개변수 갱신 전 신경망을 통해 전파된 데이터 샘플 수
# 학습률(learning rate) : 모델의 매개변수 변화율을 조절하는 보폭(너무 크면 최적값을 지나쳐버리고 너무 작으면 학습이 오래 걸림)
learning_rate = 1e-3
batch_size = 64
epochs = 5

In [ ]:
# 손실 함수(loss function) : 획득한 결과와 실제값 사이의 틀린 정도를 측정
# 학습용 데이터 제공 시 학습되지 않은 신경망은 정답을 벗어날 확률이 높기에 학습 중에 이 값을 최소화해야 함(손실 함수가 벗어난 정도를 계산).

# 모델의 출력 logit을 전달하면 logit을 정규화하여 예측 오류 계산
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# 최적화(optimizer) : 모델의 오류를 줄이기 위해 모델 매개변수를 조절하는 과정

# 모델 매개변수와 학습률을 전달해 optimizer 초기화
# 인스턴스 생성 순간 가중치와 편향이 랜덤 초기화
# optimizer.zero_grad() : 모델의 변화율을 재설정(기본적으로 변화도는 더해지기 때문에 중복 계산을 막기 위해 재설정)
# loss.backward() : 예측 손실 역전파(손실을 기반으로 변화도 계산 - 각 파라미터의 이동 방향과 그 양을)
# optimizer.step() : 역전파 단계에서 수집된 변화도로 매개변수 조정
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
# 전체 구현
def train_loop(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)
  